In [ ]:
import gsd.hoomd
import numpy as np
import freud
import matplotlib.pyplot as plt
import pandas as pd

def analyzedata():
    # Load trajectory
    traj_path = '/home/POLY/bhandarit/Desktop/mount_viper/chain_64_01/workspace/43ef74dd463bfa6f52acb921ee73e162/trajectory_bias_1.0.gsd'
    bcd = gsd.hoomd.open(traj_path, 'r')
    traj = bcd[10000:20000]
    frame = traj[0]

    # Build tables for psi and phi dihedrals
    table_psi = {}
    table_phi = {}
    dihedral_types = frame.dihedrals.types
    dihedrals_psi = frame.dihedrals.typeid == dihedral_types.index('psi')
    dihedrals_phi = frame.dihedrals.typeid == dihedral_types.index('phi')

    for idx, group in enumerate(frame.dihedrals.group):
        if dihedrals_psi[idx]:
            table_psi[group[1]] = group
        if dihedrals_phi[idx]:
            table_phi[group[2]] = group

    # Remove mismatched dihedrals
    mismatched_keys = table_phi.keys() ^ table_psi.keys()
    for key in mismatched_keys:
        table_psi.pop(key, None)
        table_phi.pop(key, None)

    # Initialize data storage (phi, psi)
    timeseries = {k: np.zeros((len(traj), 2)) for k in table_psi}

    # Dihedral angle calculator
    def compute_dihedral(array):
        b1 = array[1] - array[0]
        b2 = array[2] - array[1]
        b3 = array[3] - array[2]

        b2 /= np.linalg.norm(b2)
        n1 = np.cross(b1, b2)
        n1 /= np.linalg.norm(n1)
        n2 = np.cross(b2, b3)
        n2 /= np.linalg.norm(n2)
        m1 = np.cross(n1, b2)

        x = np.dot(n1, n2)
        y = np.dot(m1, n2)
        return -np.arctan2(y, x)

    # Compute dihedral angles
    for t, frame in enumerate(traj):
        freud_box = freud.box.Box.from_box(frame.configuration.box)
        unwrapped = freud_box.unwrap(frame.particles.position, frame.particles.image)

        for key in table_psi.keys():
            psi_group = table_psi[key]
            phi_group = table_phi[key]

            psi = compute_dihedral(unwrapped[psi_group])
            phi = compute_dihedral(unwrapped[phi_group])

            timeseries[key][t] = [phi, psi]

    # Collect all angles into lists
    all_phi = []
    all_psi = []
    for key in timeseries:
        angles = timeseries[key]
        all_phi.extend(np.degrees(angles[:, 0]))
        all_psi.extend(np.degrees(angles[:, 1]))

    # Optional: save CSV
    df = pd.DataFrame({"phi_deg": all_phi, "psi_deg": all_psi})
    csv_name = "dihedral_angles_1.0.csv"
    df.to_csv(csv_name, index=False)
    print(f" Saved dihedral angles to {csv_name}")

    # -------------------------
    #  RAMACHANDRAN PLOT
    # -------------------------
    plt.figure(figsize=(12, 10))
    hb = plt.hexbin(all_phi, all_psi, gridsize=100, cmap='viridis', bins='log')
    cbar = plt.colorbar(hb)
    cbar.set_label('Count (log scale)', fontsize=26, fontweight='bold')
    cbar.ax.tick_params(labelsize=14)
    for t in cbar.ax.get_yticklabels():
        t.set_fontweight('bold')
    plt.xlabel('Phi angle (degrees)', fontsize=30, fontweight="bold")
    plt.ylabel('Psi angle (degrees)', fontsize=30, fontweight="bold")
    plt.title('Ramachandran Plot', fontsize=26,fontweight='bold')
    plt.xlim(-180, 180)
    plt.ylim(-180, 180)
    plt.axhline(0, color='black', linestyle='--', linewidth=0.5)
    plt.axvline(0, color='black', linestyle='--', linewidth=0.5)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.xticks(fontsize=26, fontweight='bold')
    plt.yticks(fontsize=26, fontweight='bold')
    plt.tight_layout()
    plt.savefig("ramachandran_plot_1.0.png", dpi=600)
    plt.show()

# Run analysis
analyzedata()
